# Experiment 2 — Nonlinear Controlled Predictor: Shared vs Horizon-Adaptive Sources

This is the **final full-run notebook** used for the nonlinear controlled-predictor confirmation. It addresses the alternative explanation that the controlled source-utility result is merely a consequence of using a linear Ridge final predictor.

The source-selection protocol is held fixed and only the final controlled forecaster is changed from Ridge to a small nonlinear two-layer MLP.

- Source-utility teacher: unchanged Ridge residual teacher from the paper.
- Shared source set: Top-\(K(C)\) of mean utility across prediction lengths.
- Adaptive source set: Top-\(K(C)\) at the current prediction length.
- Final predictor: target history + selected source histories → two-layer MLP → endpoint value.
- Shared and Adaptive use matched architecture, initialization seed, optimizer, epoch budget, chronological split, and mini-batch order.
- Submitted run: **all 32 controlled conditions × seeds 2026, 2027, 2028** (`RUN_MODE="full"`).


In [1]:
from pathlib import Path
import gc
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import trim_mean

warnings.filterwarnings("ignore")

SEED = 2026
rng = np.random.default_rng(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)

NumPy: 1.22.3
Pandas: 1.4.1
PyTorch: 2.4.1+cu121 | device: cuda


## 1. Configuration

In [2]:
PROJECT_ROOT = Path("/data/code/2026_08")
OUTPUT_DIR = PROJECT_ROOT / "results_nonlinear_controlled_shared_vs_adaptive"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 96
PEMS_HORIZONS = [12, 24, 48]
GENERAL_HORIZONS = [96, 192, 336, 720]
MAX_TOPK = 10
MAX_TARGET_CHANNELS = 32
MAX_SOURCE_CANDIDATES = 128
MAX_TRAIN_ORIGINS = 5000
MAX_TEST_ORIGINS = 3000
TEACHER_FIT_FRACTION = 0.70
RIDGE_LAMBDA = 1e-3
SOURCE_RIDGE_LAMBDA = 1e-2

RUN_MODE = "full"     # 32 conditions x 3 matched MLP seeds
SCREEN_CONDITIONS = [
    ("Electricity", 192), ("Electricity", 336),
    ("Solar", 336), ("Solar", 720),
    ("Weather", 720),
    ("ETTh1", 336), ("ETTh1", 720), ("ETTm1", 720),
]

SCREEN_MLP_SEEDS = [2026]
FULL_MLP_SEEDS = [2026, 2027, 2028]

MLP_HIDDEN = 128
MLP_EPOCHS = 40
MLP_PATIENCE = 6
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_BATCH_SIZE = 256

print("RUN_MODE:", RUN_MODE)
print("Output:", OUTPUT_DIR)

RUN_MODE: full
Output: /data/code/2026_08/results_nonlinear_controlled_shared_vs_adaptive


## 2. Dataset protocol (same as the controlled experiment)

In [3]:
DATASET_REGISTRY = {
    "PeMS03": {
        "family": "PeMS",
        "horizons": PEMS_HORIZONS,
        "paths": [
            "/data/dataset/PeMS03.npz",
            "/data/dataset/PEMS03/PEMS03.npz",
            "/data/dataset/PEMS03.npz",
            "/data/dataset/pems/PeMS03.npz",
        ],
    },
    "PeMS04": {
        "family": "PeMS",
        "horizons": PEMS_HORIZONS,
        "paths": [
            "/data/dataset/PeMS04.npz",
            "/data/dataset/PEMS04/PEMS04.npz",
            "/data/dataset/PEMS04.npz",
            "/data/dataset/pems/PeMS04.npz",
        ],
    },
    "PeMS07": {
        "family": "PeMS",
        "horizons": PEMS_HORIZONS,
        "paths": [
            "/data/dataset/PeMS07.npz",
            "/data/dataset/PEMS07/PEMS07.npz",
            "/data/dataset/pems/PeMS07.npz",
        ],
    },
    "PeMS08": {
        "family": "PeMS",
        "horizons": PEMS_HORIZONS,
        "paths": [
            "/data/dataset/PeMS08.npz",
            "/data/dataset/PEMS08/PEMS08.npz",
            "/data/dataset/pems/PeMS08.npz",
        ],
    },
    "Electricity": {
        "family": "General",
        "horizons": GENERAL_HORIZONS,
        "paths": [
            "/data/dataset/electricity/electricity.csv",
            "/data/dataset/electricity.csv",
        ],
    },
    "Weather": {
        "family": "General",
        "horizons": GENERAL_HORIZONS,
        "paths": [
            "/data/dataset/weather/weather.csv",
            "/data/dataset/weather.csv",
        ],
    },
    "Solar": {
        "family": "General",
        "horizons": GENERAL_HORIZONS,
        "paths": [
            "/data/dataset/solar/solar_AL.txt",
            "/data/dataset/solar_AL.txt",
        ],
    },
    "ETTh1": {
        "family": "General",
        "horizons": GENERAL_HORIZONS,
        "paths": [
            "/data/dataset/ETT-small/ETTh1.csv",
            "/data/dataset/ETTh1.csv",
        ],
    },
    "ETTm1": {
        "family": "General",
        "horizons": GENERAL_HORIZONS,
        "paths": [
            "/data/dataset/ETT-small/ETTm1.csv",
            "/data/dataset/ETTm1.csv",
        ],
    },
}


def resolve_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None


rows = []

for name, spec in DATASET_REGISTRY.items():
    p = resolve_path(
        spec["paths"]
    )

    rows.append(
        {
            "dataset": name,
            "family": spec["family"],
            "path": None if p is None else str(p),
            "found": p is not None,
            "horizons": str(spec["horizons"]),
        }
    )

path_df = pd.DataFrame(rows)

display(path_df)

if not path_df["found"].all():
    print(
        "\nSome datasets are missing. "
        "Edit DATASET_REGISTRY paths before the main run."
    )


,dataset,family,path,found,horizons
0,PeMS03,PeMS,/data/dataset/PeMS03.npz,True,"[12, 24, 48]"
1,PeMS04,PeMS,/data/dataset/PeMS04.npz,True,"[12, 24, 48]"
2,PeMS07,PeMS,/data/dataset/PeMS07.npz,True,"[12, 24, 48]"
3,PeMS08,PeMS,/data/dataset/PeMS08.npz,True,"[12, 24, 48]"
4,Electricity,General,/data/dataset/electricity/electricity.csv,True,"[96, 192, 336, 720]"
5,Weather,General,/data/dataset/weather/weather.csv,True,"[96, 192, 336, 720]"
6,Solar,General,/data/dataset/solar/solar_AL.txt,True,"[96, 192, 336, 720]"
7,ETTh1,General,/data/dataset/ETT-small/ETTh1.csv,True,"[96, 192, 336, 720]"
8,ETTm1,General,/data/dataset/ETT-small/ETTm1.csv,True,"[96, 192, 336, 720]"


In [4]:
def effective_topk(C):
    return int(
        max(
            1,
            min(
                MAX_TOPK,
                math.ceil(
                    (C - 1) / 2
                ),
            ),
        )
    )


for C in [
    7,
    21,
    137,
    321,
    358,
    883,
]:
    print(
        f"C={C:4d} -> K={effective_topk(C)}"
    )


C=   7 -> K=3
C=  21 -> K=10
C= 137 -> K=10
C= 321 -> K=10
C= 358 -> K=10
C= 883 -> K=10


In [5]:
def load_series(
    dataset_name,
    path,
):
    path = Path(path)

    if path.suffix.lower() == ".npz":
        obj = np.load(path)

        if "data" in obj:
            x = obj["data"]
        else:
            x = obj[list(obj.keys())[0]]

        if x.ndim == 3:
            x = x[:, :, 0]

        if x.ndim != 2:
            raise ValueError(
                f"{dataset_name}: unexpected shape {x.shape}"
            )

        x = x.astype(np.float32)

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

        date_col = None

        for candidate in [
            "date",
            "datetime",
            "timestamp",
            "time",
        ]:
            if candidate in df.columns:
                date_col = candidate
                break

        if date_col is None:
            first = df.columns[0]

            if not pd.api.types.is_numeric_dtype(
                df[first]
            ):
                date_col = first

        if date_col is not None:
            df = df.drop(
                columns=[date_col]
            )

        df = df.select_dtypes(
            include=[np.number]
        )

        x = df.to_numpy(
            dtype=np.float32
        )

    else:
        try:
            x = np.loadtxt(
                path,
                delimiter=",",
                dtype=np.float32,
            )
        except Exception:
            x = np.loadtxt(
                path,
                dtype=np.float32,
            )

        if x.ndim == 1:
            x = x[:, None]

    if x.ndim != 2:
        raise ValueError(
            f"{dataset_name}: expected [T,C], got {x.shape}"
        )

    if not np.isfinite(x).all():
        raise ValueError(
            f"{dataset_name}: non-finite values"
        )

    return x


def split_boundaries(
    dataset_name,
    T,
):
    if dataset_name == "ETTh1":
        train_end = 12 * 30 * 24
        val_end = train_end + 4 * 30 * 24
        test_end = val_end + 4 * 30 * 24

        return (
            min(train_end, T),
            min(val_end, T),
            min(test_end, T),
        )

    if dataset_name == "ETTm1":
        unit = 30 * 24 * 4

        train_end = 12 * unit
        val_end = train_end + 4 * unit
        test_end = val_end + 4 * unit

        return (
            min(train_end, T),
            min(val_end, T),
            min(test_end, T),
        )

    return (
        int(0.70 * T),
        int(0.80 * T),
        T,
    )


def standardize_train_only(
    x,
    train_end,
):
    mean = x[
        :train_end
    ].mean(
        axis=0,
        keepdims=True,
    )

    std = x[
        :train_end
    ].std(
        axis=0,
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    return (
        (
            x - mean
        )
        /
        std
    ).astype(
        np.float32
    )


In [6]:
def evenly_subsample(
    values,
    max_n,
):
    values = np.asarray(
        values,
        dtype=np.int64,
    )

    if len(values) <= max_n:
        return values

    idx = np.linspace(
        0,
        len(values) - 1,
        max_n,
        dtype=np.int64,
    )

    return values[idx]


def make_train_origins(
    train_end,
    max_h,
):
    first = SEQ_LEN
    last = int(train_end) - int(max_h)

    if last < first:
        raise RuntimeError(
            "No valid training origins."
        )

    origins = np.arange(
        first,
        last + 1,
        dtype=np.int64,
    )

    return evenly_subsample(
        origins,
        MAX_TRAIN_ORIGINS,
    )


def make_test_origins(
    test_start,
    test_end,
    max_h,
):
    first = max(
        SEQ_LEN,
        int(test_start),
    )

    last = (
        int(test_end)
        -
        int(max_h)
    )

    if last < first:
        raise RuntimeError(
            "No valid test origins."
        )

    origins = np.arange(
        first,
        last + 1,
        dtype=np.int64,
    )

    return evenly_subsample(
        origins,
        MAX_TEST_ORIGINS,
    )


def summary_features(
    x,
    origins,
):
    origins = np.asarray(
        origins,
        dtype=np.int64,
    )

    last = x[
        origins - 1
    ]

    mean3 = np.stack(
        [
            x[t-3:t].mean(axis=0)
            for t in origins
        ],
        axis=0,
    )

    mean12 = np.stack(
        [
            x[t-12:t].mean(axis=0)
            for t in origins
        ],
        axis=0,
    )

    delta12 = (
        x[
            origins - 1
        ]
        -
        x[
            origins - 12
        ]
    )

    return np.stack(
        [
            last,
            mean3,
            mean12,
            delta12,
        ],
        axis=-1,
    ).astype(np.float32)


In [7]:
def add_bias_column(X):
    return np.concatenate(
        [
            np.ones(
                (len(X), 1),
                dtype=np.float32,
            ),
            X.astype(np.float32),
        ],
        axis=1,
    )


def ridge_fit(
    X,
    y,
    lam=RIDGE_LAMBDA,
):
    Xb = add_bias_column(
        X
    ).astype(np.float64)

    p = Xb.shape[1]

    reg = np.eye(
        p,
        dtype=np.float64,
    )

    reg[0, 0] = 0.0

    A = Xb.T @ Xb
    b = Xb.T @ y.astype(np.float64)

    return np.linalg.solve(
        A
        +
        lam
        *
        reg,
        b,
    )


def ridge_predict(
    X,
    beta,
):
    return (
        add_bias_column(X)
        @
        beta
    ).astype(np.float32)


def fit_and_mse(
    X_train,
    y_train,
    X_eval,
    y_eval,
    lam=RIDGE_LAMBDA,
):
    beta = ridge_fit(
        X_train,
        y_train,
        lam=lam,
    )

    pred = ridge_predict(
        X_eval,
        beta,
    )

    return float(
        np.mean(
            (
                y_eval
                -
                pred
            )
            **
            2
        )
    )


In [8]:
def choose_targets(C):
    if C <= MAX_TARGET_CHANNELS:
        return np.arange(
            C,
            dtype=np.int64,
        )

    return np.unique(
        np.linspace(
            0,
            C - 1,
            MAX_TARGET_CHANNELS,
            dtype=np.int64,
        )
    )


def source_candidate_map(
    train_features,
    targets,
):
    """
    Candidate pool is selected only from the currently available
    historical fit data.

    The pool itself is horizon-invariant.
    """
    current = train_features[
        :,
        :,
        0,
    ].astype(np.float64)

    current = (
        current
        -
        current.mean(
            axis=0,
            keepdims=True,
        )
    )

    denom = np.sqrt(
        np.sum(
            current
            *
            current,
            axis=0,
        )
    )

    denom = np.maximum(
        denom,
        1e-12,
    )

    normed = (
        current
        /
        denom[
            None,
            :
        ]
    )

    C = current.shape[1]

    pool_size = min(
        MAX_SOURCE_CANDIDATES,
        max(
            1,
            C - 1,
        ),
    )

    result = {}

    for i in targets:
        corr = (
            normed[:, i]
            @
            normed
        )

        corr = np.abs(
            corr
        )

        corr[i] = -np.inf

        idx = np.argpartition(
            corr,
            -pool_size,
        )[
            -pool_size:
        ]

        idx = idx[
            np.argsort(
                corr[idx]
            )[
                ::-1
            ]
        ]

        result[int(i)] = idx.astype(
            np.int64
        )

    return result


def safe_topk_indices(
    scores,
    k,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    k = min(
        int(k),
        len(scores),
    )

    if k <= 0:
        return np.array(
            [],
            dtype=np.int64,
        )

    idx = np.argpartition(
        scores,
        -k,
    )[
        -k:
    ]

    return idx[
        np.argsort(
            scores[idx]
        )[
            ::-1
        ]
    ]


def jaccard(a, b):
    a = set(
        np.asarray(a).tolist()
    )

    b = set(
        np.asarray(b).tolist()
    )

    union = a | b

    if not union:
        return 1.0

    return (
        len(a & b)
        /
        len(union)
    )


In [9]:
def predictive_utility_for_target(
    X_train_all,
    y_train,
    target_idx,
    source_ids,
):
    """
    Training-only teacher.

    Internally:
      first TEACHER_FIT_FRACTION -> fit
      remaining part -> utility scoring

    Source features are residualized against target-history features.
    """
    N = X_train_all.shape[0]

    split = int(
        TEACHER_FIT_FRACTION
        *
        N
    )

    split = min(
        max(
            split,
            32,
        ),
        N - 16,
    )

    fit_idx = np.arange(
        0,
        split,
        dtype=np.int64,
    )

    score_idx = np.arange(
        split,
        N,
        dtype=np.int64,
    )

    Xt = X_train_all[
        :,
        target_idx,
        :
    ].astype(np.float64)

    U = X_train_all[
        :,
        source_ids,
        :
    ].astype(np.float64)

    y = y_train.astype(np.float64)

    beta_t = ridge_fit(
        Xt[fit_idx],
        y[fit_idx],
        lam=RIDGE_LAMBDA,
    )

    pred_fit = ridge_predict(
        Xt[fit_idx],
        beta_t,
    ).astype(np.float64)

    pred_score = ridge_predict(
        Xt[score_idx],
        beta_t,
    ).astype(np.float64)

    r_fit = (
        y[fit_idx]
        -
        pred_fit
    )

    r_score = (
        y[score_idx]
        -
        pred_score
    )

    base_mse = float(
        np.mean(
            r_score * r_score
        )
    )

    Xt_fit_b = add_bias_column(
        Xt[fit_idx]
    ).astype(np.float64)

    Xt_score_b = add_bias_column(
        Xt[score_idx]
    ).astype(np.float64)

    U_fit = U[fit_idx]
    U_score = U[score_idx]

    S = U_fit.shape[1]
    Fdim = U_fit.shape[2]

    U_fit_flat = U_fit.reshape(
        len(fit_idx),
        S * Fdim,
    )

    A = (
        Xt_fit_b.T
        @
        Xt_fit_b
    )

    reg = np.eye(
        A.shape[0],
        dtype=np.float64,
    )

    reg[0, 0] = 0.0

    coef_u = np.linalg.solve(
        A
        +
        RIDGE_LAMBDA
        *
        reg,
        Xt_fit_b.T
        @
        U_fit_flat,
    )

    U_fit_res = (
        U_fit_flat
        -
        Xt_fit_b
        @
        coef_u
    ).reshape(
        len(fit_idx),
        S,
        Fdim,
    )

    U_score_flat = U_score.reshape(
        len(score_idx),
        S * Fdim,
    )

    U_score_res = (
        U_score_flat
        -
        Xt_score_b
        @
        coef_u
    ).reshape(
        len(score_idx),
        S,
        Fdim,
    )

    gram = np.einsum(
        "nsf,nsg->sfg",
        U_fit_res,
        U_fit_res,
    )

    cross = np.einsum(
        "nsf,n->sf",
        U_fit_res,
        r_fit,
    )

    eye = np.eye(
        Fdim,
        dtype=np.float64,
    )[
        None,
        :,
        :
    ]

    beta_s = np.linalg.solve(
        gram
        +
        SOURCE_RIDGE_LAMBDA
        *
        eye,
        cross[
            :,
            :,
            None
        ],
    )[
        :,
        :,
        0
    ]

    pred_res_score = np.einsum(
        "nsf,sf->ns",
        U_score_res,
        beta_s,
    )

    err = (
        r_score[
            :,
            None
        ]
        -
        pred_res_score
    )

    mse_source = np.mean(
        err
        *
        err,
        axis=0,
    )

    utility = (
        100.0
        *
        (
            base_mse
            -
            mse_source
        )
        /
        max(
            base_mse,
            1e-12,
        )
    )

    return {
        "source_ids": np.asarray(
            source_ids,
            dtype=np.int64,
        ),
        "utility": utility.astype(
            np.float32
        ),
        "teacher_base_mse": base_mse,
    }


In [10]:
def build_source_sets(
    utility_by_h,
    horizons,
    K,
):
    """
    Shared:
      Top-K of mean utility over horizons.

    Adaptive:
      Top-K separately for each horizon.
    """
    source_ids = utility_by_h[
        horizons[0]
    ][
        "source_ids"
    ]

    stack = np.stack(
        [
            utility_by_h[h][
                "utility"
            ]
            for h in horizons
        ],
        axis=0,
    )

    shared_score = stack.mean(
        axis=0
    )

    shared_local = safe_topk_indices(
        shared_score,
        K,
    )

    shared_sources = source_ids[
        shared_local
    ]

    adaptive_sources = {}

    diagnostics = {}

    for h in horizons:
        utility = utility_by_h[h][
            "utility"
        ]

        local = safe_topk_indices(
            utility,
            K,
        )

        sources = source_ids[
            local
        ]

        adaptive_sources[h] = sources

        diagnostics[h] = {
            "shared_adaptive_jaccard": (
                jaccard(
                    shared_sources,
                    sources,
                )
            ),
            "adaptive_utility_mean": float(
                np.mean(
                    utility[local]
                )
            ),
            "shared_utility_at_h_mean": float(
                np.mean(
                    utility[shared_local]
                )
            ),
        }

    return (
        shared_sources,
        adaptive_sources,
        diagnostics,
    )


In [11]:
def feature_matrix_for_sources(
    X_all,
    target_idx,
    source_ids,
):
    target = X_all[
        :,
        target_idx,
        :
    ]

    if len(source_ids) == 0:
        return target

    source = X_all[
        :,
        source_ids,
        :
    ].reshape(
        len(X_all),
        -1,
    )

    return np.concatenate(
        [
            target,
            source,
        ],
        axis=1,
    )


def selected_forecast_mse(
    X_fit,
    y_fit,
    X_eval,
    y_eval,
    target_idx,
    source_ids,
):
    Xtr = feature_matrix_for_sources(
        X_fit,
        target_idx,
        source_ids,
    )

    Xev = feature_matrix_for_sources(
        X_eval,
        target_idx,
        source_ids,
    )

    return fit_and_mse(
        Xtr,
        y_fit,
        Xev,
        y_eval,
        lam=RIDGE_LAMBDA,
    )


## 3. Matched nonlinear endpoint forecaster

In [12]:
def set_torch_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class EndpointMLP(nn.Module):
    def __init__(self, in_dim, hidden=MLP_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def fit_endpoint_mlp(X_train, y_train, seed):
    """Chronological 80/20 inner split; returns best validation state."""
    set_torch_seed(seed)
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    N = len(X_train)
    split = min(max(int(0.80 * N), 32), N - 16)

    Xfit, Xval = X_train[:split], X_train[split:]
    yfit, yval = y_train[:split], y_train[split:]

    mu = Xfit.mean(axis=0, keepdims=True)
    sd = np.maximum(Xfit.std(axis=0, keepdims=True), 1e-6)
    Xfit = (Xfit - mu) / sd
    Xval = (Xval - mu) / sd

    model = EndpointMLP(Xfit.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    rng_local = np.random.default_rng(seed)
    best_state, best_val, bad = None, float("inf"), 0

    for epoch in range(MLP_EPOCHS):
        model.train()
        order = rng_local.permutation(len(Xfit))
        for st in range(0, len(order), MLP_BATCH_SIZE):
            idx = order[st:st+MLP_BATCH_SIZE]
            xb = torch.from_numpy(Xfit[idx]).to(DEVICE)
            yb = torch.from_numpy(yfit[idx]).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            xv = torch.from_numpy(Xval).to(DEVICE)
            yv = torch.from_numpy(yval).to(DEVICE)
            val = F.mse_loss(model(xv), yv).item()
        if val < best_val - 1e-8:
            best_val = val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= MLP_PATIENCE:
                break

    model.load_state_dict(best_state, strict=True)
    model.eval()
    return {"model": model, "mu": mu.astype(np.float32), "sd": sd.astype(np.float32), "best_val_mse": best_val}


@torch.inference_mode()
def predict_endpoint_mlp(fitted, X):
    Xn = (np.asarray(X, dtype=np.float32) - fitted["mu"]) / fitted["sd"]
    out = []
    for st in range(0, len(Xn), 512):
        xb = torch.from_numpy(Xn[st:st+512]).to(DEVICE)
        out.append(fitted["model"](xb).detach().cpu().numpy())
    return np.concatenate(out)


def selected_forecast_mlp_mse(X_train_all, y_train, X_test_all, y_test, target_idx, source_ids, seed):
    Xtr = feature_matrix_for_sources(X_train_all, target_idx, source_ids).astype(np.float32)
    Xte = feature_matrix_for_sources(X_test_all, target_idx, source_ids).astype(np.float32)
    fitted = fit_endpoint_mlp(Xtr, y_train, seed)
    pred = predict_endpoint_mlp(fitted, Xte)
    mse = float(np.mean((np.asarray(y_test) - pred) ** 2))
    del fitted["model"]
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return mse


## 4. Run Shared vs Adaptive MLP comparison

In [13]:
def conditions_for_mode():
    if RUN_MODE == "screen":
        return SCREEN_CONDITIONS, SCREEN_MLP_SEEDS
    if RUN_MODE == "full":
        cond = []
        for d, spec in DATASET_REGISTRY.items():
            for H in spec["horizons"]:
                cond.append((d, int(H)))
        return cond, FULL_MLP_SEEDS
    raise ValueError(RUN_MODE)


def evaluate_dataset_mlp(dataset_name, requested_horizons, seeds):
    spec = DATASET_REGISTRY[dataset_name]
    path = resolve_path(spec["paths"])
    raw = load_series(dataset_name, path)
    T, C = raw.shape
    tr_end, va_end, te_end = split_boundaries(dataset_name, T)
    x = standardize_train_only(raw, tr_end)

    all_horizons = list(spec["horizons"])
    max_h = max(all_horizons)
    train_origins = make_train_origins(tr_end, max_h)
    test_origins = make_test_origins(va_end, te_end, max_h)
    X_train = summary_features(x, train_origins)
    X_test = summary_features(x, test_origins)
    targets = choose_targets(C)
    candidate_map = source_candidate_map(X_train, targets)
    K = effective_topk(C)

    rows = []
    for ti, target_idx0 in enumerate(targets):
        i = int(target_idx0)
        src = candidate_map[i]
        utility_by_h = {}
        for h in all_horizons:
            ytr = x[train_origins + h - 1, i]
            utility_by_h[h] = predictive_utility_for_target(X_train, ytr, i, src)
        shared_sources, adaptive_sources, diagnostics = build_source_sets(utility_by_h, all_horizons, K)

        for h in requested_horizons:
            ytr = x[train_origins + h - 1, i]
            yte = x[test_origins + h - 1, i]
            for seed in seeds:
                # SAME seed before each variant => matched initialization/order.
                shared_mse = selected_forecast_mlp_mse(X_train, ytr, X_test, yte, i, shared_sources, seed)
                adaptive_mse = selected_forecast_mlp_mse(X_train, ytr, X_test, yte, i, adaptive_sources[h], seed)
                gain = 100.0 * (shared_mse - adaptive_mse) / max(shared_mse, 1e-12)
                rows.append({
                    "dataset": dataset_name,
                    "horizon": int(h),
                    "target_channel": i,
                    "seed": int(seed),
                    "C": int(C), "K": int(K),
                    "shared_adaptive_jaccard": float(diagnostics[h]["shared_adaptive_jaccard"]),
                    "shared_mlp_mse": shared_mse,
                    "adaptive_mlp_mse": adaptive_mse,
                    "adaptive_gain_vs_shared_mlp_%": gain,
                    "shared_sources": ",".join(map(str, shared_sources.tolist())),
                    "adaptive_sources": ",".join(map(str, adaptive_sources[h].tolist())),
                })
        if (ti + 1) % max(1, len(targets)//4) == 0:
            print(f"  {dataset_name}: {ti+1}/{len(targets)} targets")

    return pd.DataFrame(rows)


conditions, mlp_seeds = conditions_for_mode()
by_dataset = {}
for d, h in conditions:
    by_dataset.setdefault(d, []).append(h)

frames, failures = [], []
for d, hs in by_dataset.items():
    out = OUTPUT_DIR / f"{RUN_MODE}_{d}_mlp_rows.csv"
    if out.exists():
        print("Reusing", out)
        frames.append(pd.read_csv(out))
        continue
    try:
        df = evaluate_dataset_mlp(d, sorted(set(hs)), mlp_seeds)
        df.to_csv(out, index=False)
        frames.append(df)
    except Exception as e:
        print("FAILED", d, repr(e))
        failures.append({"dataset": d, "error": repr(e)})

mlp_rows = pd.concat(frames, ignore_index=True)
mlp_rows.to_csv(OUTPUT_DIR / f"{RUN_MODE}_all_target_seed_rows.csv", index=False)
pd.DataFrame(failures).to_csv(OUTPUT_DIR / f"{RUN_MODE}_failures.csv", index=False)
print("Rows:", len(mlp_rows), "| failures:", len(failures))


  PeMS03: 8/32 targets
  PeMS03: 16/32 targets
  PeMS03: 24/32 targets
  PeMS03: 32/32 targets
  PeMS04: 8/32 targets
  PeMS04: 16/32 targets
  PeMS04: 24/32 targets
  PeMS04: 32/32 targets
  PeMS07: 8/32 targets
  PeMS07: 16/32 targets
  PeMS07: 24/32 targets
  PeMS07: 32/32 targets
  PeMS08: 8/32 targets
  PeMS08: 16/32 targets
  PeMS08: 24/32 targets
  PeMS08: 32/32 targets
  Electricity: 8/32 targets
  Electricity: 16/32 targets
  Electricity: 24/32 targets
  Electricity: 32/32 targets
  Weather: 5/21 targets
  Weather: 10/21 targets
  Weather: 15/21 targets
  Weather: 20/21 targets
  Solar: 8/32 targets
  Solar: 16/32 targets
  Solar: 24/32 targets
  Solar: 32/32 targets
  ETTh1: 1/7 targets
  ETTh1: 2/7 targets
  ETTh1: 3/7 targets
  ETTh1: 4/7 targets
  ETTh1: 5/7 targets
  ETTh1: 6/7 targets
  ETTh1: 7/7 targets
  ETTm1: 1/7 targets
  ETTm1: 2/7 targets
  ETTm1: 3/7 targets
  ETTm1: 4/7 targets
  ETTm1: 5/7 targets
  ETTm1: 6/7 targets
  ETTm1: 7/7 targets
Rows: 2340 | failures

## 5. Condition-level summary and decision

In [14]:
# First average matched MLP seeds per target, then pool target MSE within each condition.
target_mean = (
    mlp_rows.groupby(["dataset", "horizon", "target_channel"], as_index=False)
    .agg(
        shared_mlp_mse=("shared_mlp_mse", "mean"),
        adaptive_mlp_mse=("adaptive_mlp_mse", "mean"),
        target_gain=("adaptive_gain_vs_shared_mlp_%", "mean"),
        shared_adaptive_jaccard=("shared_adaptive_jaccard", "first"),
    )
)

condition_summary = (
    target_mean.groupby(["dataset", "horizon"], as_index=False)
    .agg(
        n_targets=("target_channel", "size"),
        shared_mlp_mse=("shared_mlp_mse", "mean"),
        adaptive_mlp_mse=("adaptive_mlp_mse", "mean"),
        target_win_rate=("target_gain", lambda s: float(np.mean(np.asarray(s) > 0))),
        median_target_gain=("target_gain", "median"),
        mean_set_jaccard=("shared_adaptive_jaccard", "mean"),
    )
)
condition_summary["pooled_adaptive_gain_%"] = 100.0 * (
    condition_summary["shared_mlp_mse"] - condition_summary["adaptive_mlp_mse"]
) / np.maximum(condition_summary["shared_mlp_mse"], 1e-12)
condition_summary["adaptive_better"] = (condition_summary["pooled_adaptive_gain_%"] > 0).astype(int)
condition_summary.to_csv(OUTPUT_DIR / f"{RUN_MODE}_condition_summary.csv", index=False)

display(condition_summary.round(4))
n = len(condition_summary)
print("Adaptive MLP > Shared MLP:", int(condition_summary["adaptive_better"].sum()), "/", n)
print("Mean pooled MLP gain:", f'{condition_summary["pooled_adaptive_gain_%"].mean():+.3f}%')
print("Median pooled MLP gain:", f'{condition_summary["pooled_adaptive_gain_%"].median():+.3f}%')

print("\\nPaper-use guide:")
print("- Strongest outcome: Adaptive remains positive across many conditions under the nonlinear MLP.")
print("  This directly weakens the 'Ridge-only artifact' explanation.")
print("- If the MLP effect disappears, do NOT add it as confirmatory evidence; instead keep the current")
print("  scope statement that P is operationally defined by the sparse Ridge protocol.")


,dataset,horizon,n_targets,shared_mlp_mse,adaptive_mlp_mse,target_win_rate,median_target_gain,mean_set_jaccard,pooled_adaptive_gain_%,adaptive_better
0,ETTh1,96,7,0.6983,0.6790,0.2857,-3.5801,0.4714,2.7592,1
1,ETTh1,192,7,0.7549,0.7490,0.2857,0.0000,0.7857,0.7793,1
2,ETTh1,336,7,0.8219,0.7633,0.4286,0.0000,0.7429,7.1191,1
3,ETTh1,720,7,1.1126,1.0099,0.2857,-1.5557,0.7857,9.2289,1
4,ETTm1,96,7,0.5968,0.5723,0.5714,0.0869,0.6286,4.1039,1
5,ETTm1,192,7,0.5442,0.5167,0.8571,2.9446,0.6000,5.0467,1
6,ETTm1,336,7,1.1892,1.0502,0.8571,1.2370,0.6714,11.6845,1
7,ETTm1,720,7,1.1952,1.1036,0.2857,0.0000,0.8857,7.6633,1
8,Electricity,96,32,0.2673,0.2644,0.4375,-1.5788,0.4772,1.0749,1
9,Electricity,192,32,0.2528,0.2529,0.4062,-0.9979,0.5199,-0.0093,0


Adaptive MLP > Shared MLP: 19 / 32
Mean pooled MLP gain: +1.667%
Median pooled MLP gain: +1.154%
\nPaper-use guide:
- Strongest outcome: Adaptive remains positive across many conditions under the nonlinear MLP.
  This directly weakens the 'Ridge-only artifact' explanation.
- If the MLP effect disappears, do NOT add it as confirmatory evidence; instead keep the current
  scope statement that P is operationally defined by the sparse Ridge protocol.


In [15]:

# ============================================================
# Full-run confirmation and seed-stability summary
# ============================================================
expected_conditions = 32
expected_seeds = 3

print("Unique conditions:", condition_summary[["dataset", "horizon"]].drop_duplicates().shape[0])
print("Expected conditions:", expected_conditions)
print("Unique seeds:", sorted(mlp_rows["seed"].unique().tolist()))

if condition_summary[["dataset", "horizon"]].drop_duplicates().shape[0] != expected_conditions:
    print("WARNING: fewer than 32 conditions were summarized.")
if mlp_rows["seed"].nunique() != expected_seeds:
    print("WARNING: fewer than 3 MLP seeds were found.")

seed_condition = (
    mlp_rows.groupby(["seed", "dataset", "horizon"], as_index=False)
    .agg(
        shared_mlp_mse=("shared_mlp_mse", "mean"),
        adaptive_mlp_mse=("adaptive_mlp_mse", "mean"),
    )
)
seed_condition["pooled_adaptive_gain_%"] = 100.0 * (
    seed_condition["shared_mlp_mse"] - seed_condition["adaptive_mlp_mse"]
) / seed_condition["shared_mlp_mse"].clip(lower=1e-12)

seed_summary = (
    seed_condition.groupby("seed", as_index=False)
    .agg(
        n_conditions=("pooled_adaptive_gain_%", "size"),
        wins=("pooled_adaptive_gain_%", lambda s: int((s > 0).sum())),
        mean_gain=("pooled_adaptive_gain_%", "mean"),
        median_gain=("pooled_adaptive_gain_%", "median"),
    )
)
seed_summary["win_rate"] = seed_summary["wins"] / seed_summary["n_conditions"]
display(seed_summary.round(4))

seed_summary.to_csv(
    OUTPUT_DIR / "full_seed_stability_summary.csv",
    index=False,
)

print("\nFiles to send for analysis:")
print(OUTPUT_DIR / "full_condition_summary.csv")
print(OUTPUT_DIR / "full_seed_stability_summary.csv")
print(OUTPUT_DIR / "full_all_target_seed_rows.csv")


Unique conditions: 32
Expected conditions: 32
Unique seeds: [2026, 2027, 2028]


,seed,n_conditions,wins,mean_gain,median_gain,win_rate
0,2026,32,21,1.3538,2.1618,0.6562
1,2027,32,18,1.8958,0.3465,0.5625
2,2028,32,19,1.7293,1.2016,0.5938



Files to send for analysis:
/data/code/2026_08/results_nonlinear_controlled_shared_vs_adaptive/full_condition_summary.csv
/data/code/2026_08/results_nonlinear_controlled_shared_vs_adaptive/full_seed_stability_summary.csv
/data/code/2026_08/results_nonlinear_controlled_shared_vs_adaptive/full_all_target_seed_rows.csv
